In [10]:
import sys
sys.path.append("..")  # Adds the parent directory (ai-service) to the Python path

from utils.preprocess import preprocess_image


In [ ]:
pip install transformers==4.41.2


In [8]:
pip install torch

Note: you may need to restart the kernel to use updated packages.


In [9]:
from tensorflow.keras.models import load_model

# Load the model using the relative path from the notebooks directory
model = load_model("../model/usg_model.keras")


In [ ]:
from transformers import pipeline
pipe = pipeline("text-generation", model="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")


d:\java full stack project\OVAHealth AI\python ai microservices\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus\.cache\huggingface\hub\models--deepseek-ai--DeepSeek-R1-Distill-Qwen-1.5B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Special tokens have been added in the vocabulary, make sure

In [11]:
result = pipe("What is PCOS in terms of menstruation")

print(result)

d:\java full stack project\OVAHealth AI\python ai microservices\venv\Lib\site-packages\transformers\generation\utils.py:1168: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


[{'generated_text': 'What is PCOS in terms of menstruation?\nPCOS stands for Prostate Cosmesis,'}]


In [12]:
def generate_pcos_prompt(patient_data):
    symptoms = []

    if patient_data["Weight gain(Y/N)"] == "Y":
        symptoms.append("experiencing weight gain")
    if patient_data["hair growth(Y/N)"] == "Y":
        symptoms.append("having excessive hair growth (hirsutism)")
    if patient_data["Skin darkening (Y/N)"] == "Y":
        symptoms.append("noticing skin darkening")
    if patient_data["Hair loss(Y/N)"] == "Y":
        symptoms.append("experiencing hair loss")
    if patient_data["Pimples(Y/N)"] == "Y":
        symptoms.append("having frequent pimples/acne")
    if patient_data["Fast food (Y/N)"] == "Y":
        symptoms.append("consuming fast food regularly")
    if patient_data["Reg.Exercise(Y/N)"] == "N":
        symptoms.append("not engaging in regular exercise")

    # BMI Classification
    bmi = patient_data["BMI"]
    if bmi >= 25:
        symptoms.append("is overweight")
    elif bmi < 18.5:
        symptoms.append("is underweight")

    # Menstrual Irregularities
    if patient_data["Cycle(R/I)"] == "I":
        symptoms.append("has irregular menstrual cycles")

    # Structuring the prompt
    patient_summary = ", ".join(symptoms) if symptoms else "has no significant symptoms reported"
    
    prompt = (f"The patient {patient_summary}. Based on these symptoms, "
              "could this be Polycystic Ovary Syndrome (PCOS)? Are further tests such as ultrasound or hormone analysis recommended? Please answer concisely in one line.")
    
    return prompt

# Example usage with a patient dictionary
patient_data = {
    "BMI": 27.5,
    "Weight gain(Y/N)": "Y",
    "hair growth(Y/N)": "Y",
    "Skin darkening (Y/N)": "N",
    "Hair loss(Y/N)": "Y",
    "Pimples(Y/N)": "Y",
    "Fast food (Y/N)": "Y",
    "Reg.Exercise(Y/N)": "N",
    "Cycle(R/I)": "I",
}

pcos_prompt = generate_pcos_prompt(patient_data)
print(pcos_prompt)

The patient experiencing weight gain, having excessive hair growth (hirsutism), experiencing hair loss, having frequent pimples/acne, consuming fast food regularly, not engaging in regular exercise, is overweight, has irregular menstrual cycles. Based on these symptoms, could this be Polycystic Ovary Syndrome (PCOS)? Are further tests such as ultrasound or hormone analysis recommended? Please answer concisely in one line.


In [1]:
def predict_ultrasound(image_path):

    # preprocess image
    processed_image = preprocess_image(image_path)

    # CNN prediction
    prediction = model.predict(processed_image)

    raw_score = float(prediction[0][0])

    infected_probability = raw_score
    normal_probability = 1 - raw_score

    # =========================
    # DETERMINE RESULT
    # =========================

    if infected_probability >= 0.5:

        result = "PCOS Detected"

        confidence = infected_probability * 100

        prompt = f"""
        A pelvic ultrasound scan was analyzed using AI.

        Result: PCOS Detected
        Confidence: {round(confidence,2)}%

        Explain medically why this scan may indicate PCOS.
        Mention follicles, ovarian morphology, and possible hormonal imbalance.
        Keep explanation simple and concise.
        """

    else:

        result = "Normal"

        confidence = normal_probability * 100

        prompt = f"""
        A pelvic ultrasound scan was analyzed using AI.

        Result: Normal
        Confidence: {round(confidence,2)}%

        Explain medically why the ovaries appear normal.
        Keep explanation simple and concise.
        """

    # =========================
    # GENERATE AI EXPLANATION
    # =========================

    response = pipe(
        prompt,
        max_new_tokens=120
    )

    explanation = response[0]["generated_text"]

    # =========================
    # RETURN FINAL RESPONSE
    # =========================

    return {

        "prediction": result,

        "confidence": round(confidence, 2),

        "raw_prediction": round(raw_score, 4),

        "explanation": explanation
    }